# Retrieval-Only


# 1. Setup

In [1]:
from uretriever.configs.path_config import DATA_DIR, MODEL_DIR, PROMPT_DIR


RAW_DATA_DIR = DATA_DIR / "raw"
INDEX_DATA_DIR = DATA_DIR / "processed"
INDEX_DATA_DIR.mkdir(parents=True, exist_ok=True)

# CSV corpus files for index building
LAWS_CSV_PATH = RAW_DATA_DIR / "laws_de.csv"
COURTS_CSV_PATH = RAW_DATA_DIR / "court_considerations.csv"
# BM25 Index files
LAWS_BM25_PATH = INDEX_DATA_DIR / "laws_bm25_index.pkl"
COURTS_BM25_PATH = INDEX_DATA_DIR / "courts_bm25_index.pkl"
# Embedding Index files
LAWS_EMBED_PATH = INDEX_DATA_DIR / "laws_embedding_index.pkl"
COURTS_EMBED_PATH = INDEX_DATA_DIR / "courts_embedding_index.pkl"
FORCE_REBUILD = False

# Retrieval settings
TOP_K = 20

print(f"RAW_DATA_DIR: {RAW_DATA_DIR}")
print(f"INDEX_DATA_DIR: {INDEX_DATA_DIR}")
print(f"PROMPT_DIR: {PROMPT_DIR}")

RAW_DATA_DIR: D:\LegalRetrievalAgent\data\raw
INDEX_DATA_DIR: D:\LegalRetrievalAgent\data\processed
PROMPT_DIR: D:\LegalRetrievalAgent\prompts


# 2. Build or Load BM25 Indices

In [2]:
# Load or build BM25 index
from uretriever.BM25Index import build_bm25_index


laws_bm25 = build_bm25_index(
    name="laws",
    csv_path=LAWS_CSV_PATH,
    index_path=LAWS_BM25_PATH,
    force_rebuild=FORCE_REBUILD,
)
print(f"\nLaws index: {len(laws_bm25.documents):,} documents")

Loading cached laws index from D:\LegalRetrievalAgent\data\processed\laws_bm25_index.pkl
  Loaded 175,933 documents

Laws index: 175,933 documents


In [3]:
# Test search
test_results = laws_bm25.search("Vertrag", top_k=3)
print(f"\nTest search 'Vertrag': {len(test_results)} results")
if test_results:
    print(f"  Top result: {test_results[0].get('citation', 'N/A')}")


Test search 'Vertrag': 3 results
  Top result: Art. 17 Abs. 6 VID


In [4]:
# Load or build courts index
courts_bm25 = build_bm25_index(
    name="courts",
    csv_path=COURTS_CSV_PATH,
    index_path=COURTS_BM25_PATH,
    force_rebuild=FORCE_REBUILD,
    max_rows=100000  # Change to use bigger corpus
)
print(f"\nCourts index: {len(courts_bm25.documents):,} documents")

Loading cached courts index from D:\LegalRetrievalAgent\data\processed\courts_bm25_index.pkl
  Loaded 100,000 documents

Courts index: 100,000 documents


In [5]:
# Test search
test_results = courts_bm25.search("Meinungsfreiheit", top_k=3)
print(f"\nTest search 'Meinungsfreiheit': {len(test_results)} results")
if test_results:
    print(f"  Top result: {test_results[0].get('citation', 'N/A')}")


Test search 'Meinungsfreiheit': 3 results
  Top result: BGE 148 I 33 E. 6.1


# 3. Build or Load Embedding Indices

In [6]:
# Load or build Embedding index
import torch

from uretriever.EmbeddingIndex import build_embedding_index

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {device}")

laws_embed = build_embedding_index(
    name="laws",
    csv_path=LAWS_CSV_PATH,
    index_path=LAWS_EMBED_PATH,
    force_rebuild=FORCE_REBUILD,
    model_name="intfloat/multilingual-e5-base",
    device=device

)
print(f"\nLaws index: {len(laws_embed.documents):,} documents")


Using device: cuda
Loading cached laws embedding index from D:\LegalRetrievalAgent\data\processed\laws_embedding_index.pkl


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Loaded 175,933 documents

Laws index: 175,933 documents


In [7]:
# Test search
test_results = laws_embed.search("Vertrag", top_k=3)
print(f"\nTest search 'Vertrag': {len(test_results)} results")
if test_results:
    print(f"  Top result: {test_results[0].get('citation', 'N/A')}")


Test search 'Vertrag': 3 results
  Top result: Art. 22 Abs. 1 OR


In [8]:
courts_embed = build_embedding_index(
    name="laws",
    csv_path=COURTS_CSV_PATH,
    index_path=COURTS_EMBED_PATH,
    force_rebuild=FORCE_REBUILD,
    model_name="intfloat/multilingual-e5-base",
    device=device,
    max_rows=100000  # Change to use bigger corpus
)
print(f"\Courts index: {len(courts_embed.documents):,} documents")

Loading cached laws embedding index from D:\LegalRetrievalAgent\data\processed\courts_embedding_index.pkl


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Loaded 100,000 documents
\Courts index: 100,000 documents


In [9]:
# Test search
test_results = courts_embed.search("Meinungsfreiheit", top_k=3)
print(f"\nTest search 'Meinungsfreiheit': {len(test_results)} results")
if test_results:
    print(f"  Top result: {test_results[0].get('citation', 'N/A')}")


Test search 'Meinungsfreiheit': 3 results
  Top result: BGE 134 IV 216 E. 5.3


# 4. Evaluation

In [10]:
import pandas as pd

valid_df = pd.read_csv(RAW_DATA_DIR / "val.csv")
valid_df.head()

,query_id,query,gold_citations
0,val_001,May a court lawfully order a three‑month exten...,Art. 221 Abs. 1 StPO;Art. 140 Abs. 1 StGB;Art....
1,val_002,A claimant holding a national vocational diplo...,Art. 8 Abs. 1 ATSG;Art. 8 Abs. 1 IVG;Art. 17 A...
2,val_003,"A. Rivera, a Peruvian national born in 1994 an...",Art. 29 Abs. 2 BV;Art. 221 Abs. 1 StPO;Art. 39...
3,val_004,"Mr. Dalton, born in 1941 and resident in a sma...",Art. 505 Abs. 1 ZGB;Art. 467 ZGB;Art. 469 Abs....
4,val_005,"A parent, separated from their co-parent since...",Art. 133 Abs. 1 ZGB;Art. 133 Abs. 2 ZGB;Art. 2...


In [11]:
from tqdm import tqdm

from uretriever.hybride_retriever import hybrid_search


def generate_pred_df(
        df: pd.DataFrame, 
        laws_bm25, 
        courts_bm25, 
        laws_embed, 
        courts_embed, 
        alpha: float,
        beta: float,
        top_k: int
    ) -> pd.DataFrame:
    preds = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Generating predictions"):
        query_id = row["query_id"]
        query = row["query"]

        law_results = hybrid_search(query, laws_bm25, laws_embed, alpha=alpha, top_k=int(top_k * beta))
        courts_results = hybrid_search(query, courts_bm25, courts_embed, alpha=alpha, top_k=top_k - int(top_k * beta))

        results = sorted(law_results + courts_results, key=lambda x: x['_score'], reverse=True)
        results = results[:top_k]

        citations = [result["citation"] for result in results]
        citations_str = ";".join(citations) if citations else " "

        preds.append({
            "query_id": query_id,
            "predicted_citations": citations_str,
        })

    pred_df = pd.DataFrame(preds)
    return pred_df

In [12]:
# Test with a single query
query = "What are the requirements for a valid contract under Swiss law?"
law_results = hybrid_search(query, laws_bm25, laws_embed, alpha=0.3, top_k=5)
courts_results = hybrid_search(query, courts_bm25, courts_embed, alpha=0.3, top_k=5)
print(law_results)
print(courts_results)

[{'citation': 'Art. 15 Abs. 1 VAG', 'text': '1 Ein ausländisches Versicherungsunternehmen, das beabsichtigt, in der Schweiz eine Versicherungstätigkeit aufzunehmen, muss neben den Voraussetzungen nach den Artikeln 7–14a folgende Voraussetzungen erfüllen:a. Es ist in seinem Sitzstaat zur Ausübung der Versicherungstätigkeit befugt.\nb. Es errichtet in der Schweiz eine Niederlassung, lässt sie ins Handelsregister eintragen und bestellt als deren Leiterin oder Leiter eine Generalbevollmächtigte oder einen Generalbevollmächtigten.\nc. Es verfügt am Hauptsitz über ein Kapital nach Artikel 8 und weist eine auch die Geschäftstätigkeit in der Schweiz umfassende ausreichende Solvabilität im Sinne der Artikel 9–9c aus.\nd. Es verfügt in der Schweiz über einen Organisationsfonds nach Artikel 10 und entsprechende Vermögenswerte.\ne. Es hinterlegt in der Schweiz eine Kaution, die einem bestimmten Bruchteil des inländischen Geschäftsvolumens entspricht.', '_bm25': 0.0, '_embed': 1.0, '_score': 0.7}, 

In [13]:
from uretriever.citation import parse_citations
from uretriever.metrics import macro_f1


for alpha in [0, 0.3, 1]:
    print(f"\n-- Alpha: {alpha} --\n")

    pred_df = generate_pred_df(
        valid_df,
        laws_bm25=laws_bm25,
        courts_bm25=courts_bm25,
        laws_embed=laws_embed,
        courts_embed=courts_embed,
        alpha=alpha,
        beta=0.5,
        top_k=TOP_K
    )
 
    merged_df = pd.merge(pred_df, valid_df, on="query_id", how="inner")

    preds = [
        parse_citations(row.get("predicted_citations", "")) 
        for _, row in merged_df.iterrows()
    ]
    golds = [
        parse_citations(row.get("gold_citations", "")) 
        for _, row in merged_df.iterrows()
    ]

    scores = macro_f1(preds, golds)
    print("\n-- Scores --\n")
    print(scores)


-- Alpha: 0 --



Generating predictions: 100%|██████████| 10/10 [06:10<00:00, 37.02s/it]



-- Scores --

{'macro_precision': 0.005, 'macro_recall': 0.01, 'macro_f1': 0.006666666666666666}

-- Alpha: 0.3 --



Generating predictions: 100%|██████████| 10/10 [04:33<00:00, 27.37s/it]



-- Scores --

{'macro_precision': 0.005, 'macro_recall': 0.01, 'macro_f1': 0.006666666666666666}

-- Alpha: 1 --



Generating predictions: 100%|██████████| 10/10 [05:09<00:00, 30.91s/it]


-- Scores --

{'macro_precision': 0.0, 'macro_recall': 0.0, 'macro_f1': 0.0}
